# PrediCare TPA – RAG v3 (multi-documents)

Pipeline complet : **PDF → nettoyage → chunking par structure → index hybride → reranking → réponse rédigée**.

**Pour ajouter un document :** déposez le PDF dans le dossier, puis relancez les cellules à partir de la section 1 (ou seulement les sections 1 à 4).

| Problème v1 | Correction |
|---|---|
| Découpage par caractères | Découpage **par structure** (article / section), tableaux lus **ligne par ligne** |
| Éléments hors articles perdus | Section d'introduction conservée (titre, assureur, préambule) |
| Bruit (`DOCUMENT FICTIF`, `Page N`) | Nettoyage avant découpage, pages gardées en métadonnées |
| Expressions hors document ("unité monétaire") | Glossaire de synonymes + **bge-m3** + **BM25** + **reranker** |
| Un seul PDF | **Tous les PDF du dossier**, source citée à chaque réponse |
| Pas de mesure | Jeu de tests hit@1 / hit@3 |
| Pas de réponse rédigée | Génération (Qwen local gratuit ou API Claude) |

## 0. Installation et configuration

In [1]:
!pip install -q pymupdf sentence-transformers rank_bm25 transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 64.4 MB/s eta 0:00:00


In [2]:
import os
DOSSIER_PDF = "/content/DOSSIER_PDF"        # dossier contenant les PDF (sur Colab : "/content")
print("Dossier lu :", os.path.abspath(DOSSIER_PDF))
print("Fichiers :", sorted(os.listdir(DOSSIER_PDF))[:20])

# (Optionnel, Colab) pour téléverser des PDF depuis votre ordinateur :
# from google.colab import files
# for nom in files.upload(): print("reçu :", nom)

Dossier lu : /content/DOSSIER_PDF
Fichiers : ['Bareme_Reference_Actes.pdf', 'Conditions_Generales_CONFORT.pdf', 'Conditions_Generales_ESSENTIEL.pdf', 'Conditions_Generales_PREMIUM.pdf', 'Guide_Reseau_Soins_Antananarivo.pdf', 'Guide_Reseau_Soins_National.pdf']


## 1. Lecture propre des PDF (texte + tableaux)

In [3]:
import fitz, re, os, glob, hashlib, pickle, unicodedata, numpy as np
from collections import defaultdict, Counter

RE_BRUIT = re.compile(r"^(DOCUMENT FICTIF.*|GUIDE DU R[ÉE]SEAU\s*[–-]\s*DONN[ÉE]ES FICTIVES.*|Page\s+\d+(\s*/\s*\d+)?)\s*$", re.I)
RE_TITRE = re.compile(r"^Article\s+(\d+)\s*[–—\-:.]\s*(.+)$")
RE_PUCE = re.compile(r"^[•\-\*·▪]\s*")

def reflow(texte):
    """Recolle les lignes coupées par le PDF ; garde titres et puces sur leur propre ligne."""
    sorties, buf = [], []
    def flush():
        if buf:
            sorties.append(" ".join(buf).strip())
            buf.clear()
    for ligne in texte.split("\n"):
        l = ligne.strip()
        if not l or RE_BRUIT.match(l):
            continue
        if RE_TITRE.match(l):
            flush(); sorties.append(l)
        elif RE_PUCE.match(l):
            flush(); buf.append(l)
        else:
            buf.append(l)
    flush()
    return sorties

def nettoyer_cellule(c):
    return re.sub(r"\s+", " ", (c or "")).strip()

def trouver_tableaux(page):
    def essai(strategie):
        try:
            ts = page.find_tables(strategy=strategie).tables
        except Exception:
            ts = []
        return [t for t in ts if t.row_count >= 2 and t.col_count >= 2]
    return essai("lines")            # uniquement les vrais tableaux à bordures

def extraire_elements(chemin):
    """Texte et tableaux d'un PDF, dans l'ordre de lecture."""
    d, els = fitz.open(chemin), []
    for numero, page in enumerate(d, start=1):
        valides = []
        for t in trouver_tableaux(page):
            rows = [[nettoyer_cellule(c) for c in r] for r in t.extract()]
            rows = [r for r in rows if any(r)]
            # un vrai tableau a un en-tête complet et court ; sinon c'est du texte mis en page
            if len(rows) >= 2 and all(rows[0]) and all(len(c) <= 40 for c in rows[0]):
                valides.append((t, rows))
        zones = [fitz.Rect(t.bbox) for t, _ in valides]
        for t, rows in valides:
            els.append({"page": numero, "y0": t.bbox[1], "type": "table", "rows": rows})
        for b in page.get_text("blocks"):
            x0, y0, x1, y1, txt, _, typ = b[:7]
            if typ != 0:
                continue
            r = fitz.Rect(x0, y0, x1, y1)
            if r.get_area() > 0 and any((r & z).get_area() > 0.5 * r.get_area() for z in zones):
                continue
            paras = reflow(txt)
            if paras:
                els.append({"page": numero, "y0": y0, "type": "texte", "paras": paras})
    return sorted(els, key=lambda e: (e["page"], e["y0"]))

def est_titre_section(par):
    """Titre court en MAJUSCULES (ex. 'PHARMACIES') ou de 1 à 3 mots (ex. 'Médicaments')."""
    if len(par) > 70 or re.search(r"[.:;=]", par):
        return False
    if sum(c.isalpha() for c in par) < 4:
        return False
    base = re.sub(r"\(.*?\)", "", par).strip()          # ignore le contenu entre parenthèses
    return base.isupper() or (len(par.split()) <= 3 and par[0].isupper() and not any(ch.isdigit() for ch in par))

def construire_sections(els):
    """Documents à articles : un bloc par 'Article N – Titre'.
    Autres documents (guides, barèmes) : un bloc par titre de rubrique (PHARMACIES, SPÉCIALISTES...)."""
    mode_articles = any(RE_TITRE.match(p) for el in els if el["type"] == "texte" for p in el["paras"])
    secs, cur = [], {"numero": "0", "titre": "Début du document", "blocs": [], "pages": set()}
    for el in els:
        if el["type"] == "texte":
            for par in el["paras"]:
                m = RE_TITRE.match(par)
                if m:
                    secs.append(cur)
                    cur = {"numero": m.group(1), "titre": m.group(2).strip(), "blocs": [], "pages": {el["page"]}}
                elif not mode_articles and est_titre_section(par):
                    secs.append(cur)
                    cur = {"numero": "", "titre": par.strip("* "), "blocs": [], "pages": {el["page"]}}
                else:
                    cur["blocs"].append(("texte", par)); cur["pages"].add(el["page"])
        else:
            cur["blocs"].append(("table", el["rows"])); cur["pages"].add(el["page"])
    secs.append(cur)
    return [s for s in secs if s["blocs"]]

DOCUMENTS = {}      # nom du document -> liste de sections
DOC_HASH = {}       # nom du document -> empreinte du fichier (détecte les mises à jour)
pdfs = sorted(p for p in glob.glob(os.path.join(DOSSIER_PDF, "*")) if p.lower().endswith(".pdf"))
if not pdfs:
    tous = [f for f in glob.glob(os.path.join(DOSSIER_PDF, "*")) if os.path.isfile(f)]
    raise FileNotFoundError(
        f"Aucun PDF dans {os.path.abspath(DOSSIER_PDF)}. Fichiers vus : {[os.path.basename(f) for f in tous][:15]}. "
        "Téléversez le PDF (icône dossier à gauche, ou cellule files.upload() de la section 0) "
        "ou corrigez DOSSIER_PDF.")

for chemin in pdfs:
    nom = os.path.splitext(os.path.basename(chemin))[0].replace("_", " ")
    secs = construire_sections(extraire_elements(chemin))
    DOCUMENTS[nom] = secs
    DOC_HASH[nom] = hashlib.sha1(open(chemin, "rb").read()).hexdigest()
    print(f"\n=== {nom} : {len(secs)} sections ===")
    for s in secs:
        t = [b[0] for b in s["blocs"]]
        print(f"  Art. {s['numero'] or '-':>2} | {s['titre'][:45]:45} | p. {sorted(s['pages'])} | {t.count('texte')} para., {t.count('table')} tableau(x)")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.

=== Bareme Reference Actes : 2 sections ===
  Art.  - | BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS  | p. [1, 2, 3] | 1 para., 3 tableau(x)
  Art.  - | Médicaments                                   | p. [3, 4] | 0 para., 2 tableau(x)

=== Conditions Generales CONFORT : 15 sections ===
  Art.  0 | Début du document                             | p. [1] | 4 para., 0 tableau(x)
  Art.  1 | Objet du contrat                              | p. [1] | 1 para., 0 tableau(x)
  Art.  2 | Définitions                                   | p. [1] | 7 para., 0 tableau(x)
  Art.  3 | Adhésion et bénéficiaires                     | p. [1] | 2 para., 0 tableau(x)
  Art.  4 | Prise d'effet, durée et résiliation           | p. [1] | 1 para., 0 tableau(x)
  Art.  5 | Tableau des garanties                         | p. [1] | 1 para., 1 tableau(x)
  Art.  6 | Délais de carence                             | p. [2] | 1 para., 0

## 2. Chunking par structure
- Un article court reste **entier** ; un article long est regroupé par paragraphes/puces (jamais coupé au milieu d'une phrase).
- **Chaque ligne de tableau** devient un chunk autonome (« Imagerie — Taux : 80 % ; Plafond annuel : 500 000 Ar ; Carence : 30 jours »).
- Chaque chunk porte le **nom du document** et l'**article** ; `parent` garde l'article complet pour le LLM.

In [4]:
MAX_CHARS = 900

def decouper_texte(paras, limite=MAX_CHARS):
    unites = []
    for p in paras:
        if len(p) <= limite:
            unites.append(p)
        else:
            cur = ""
            for phrase in re.split(r"(?<=[.;])\s+", p):
                if cur and len(cur) + len(phrase) + 1 > limite:
                    unites.append(cur); cur = phrase
                else:
                    cur = (cur + " " + phrase).strip()
            if cur:
                unites.append(cur)
    groupes, cur = [], ""
    for u in unites:
        if cur and len(cur) + len(u) + 1 > limite:
            groupes.append(cur); cur = u
        else:
            cur = (cur + "\n" + u) if cur else u
    if cur:
        groupes.append(cur)
    return groupes

def lignes_tableau(rows):
    """Une ligne de tableau -> une phrase autonome. La colonne de numérotation (N°) est ignorée."""
    entetes, sorties = rows[0], []
    off = 1 if entetes[0].strip().lower() in ("n°", "no", "n", "#", "num") else 0
    heads = entetes[off:]
    for r in rows[1:]:
        r = r[off:] + [""] * (len(heads) - len(r[off:]))
        details = " ; ".join(f"{h} : {v}" for h, v in zip(heads[1:], r[1:]) if v)
        sorties.append(f"{r[0]} — {details}" if details else r[0])
    return sorties

def entete(nom, s):
    if s["numero"].isdigit() and s["numero"] != "0":
        return f"{nom} – Article {s['numero']} – {s['titre']}"
    return f"{nom} – {s['titre']}"

def texte_parent(nom, s):
    out = [entete(nom, s)]
    for kind, c in s["blocs"]:
        out += [c] if kind == "texte" else lignes_tableau(c)
    return "\n".join(out)

CHUNKS = []
def ajouter_chunk(nom, s, corps, typ):
    CHUNKS.append({
        "id": len(CHUNKS),
        "text": f"{entete(nom, s)}\n{corps}",
        "parent": texte_parent(nom, s),
        "meta": {"source": nom, "article_numero": s["numero"], "article_titre": entete(nom, s),
                 "pages": sorted(s["pages"]), "type": typ},
    })

for nom, secs in DOCUMENTS.items():
    for s in secs:
        attente = []
        for kind, c in s["blocs"]:
            if kind == "texte":
                attente.append(c)
            else:
                for g in decouper_texte(attente):
                    ajouter_chunk(nom, s, g, "texte")
                attente = []
                lignes = lignes_tableau(c)
                for l in lignes:
                    ajouter_chunk(nom, s, l, "ligne_tableau")
                if len(lignes) <= 12:        # vue d'ensemble seulement pour les petits tableaux
                    ajouter_chunk(nom, s, "\n".join(lignes), "tableau_complet")
        for g in decouper_texte(attente):
            ajouter_chunk(nom, s, g, "texte")

print("Documents :", len(DOCUMENTS), "| Chunks :", len(CHUNKS))
print(Counter(c["meta"]["type"] for c in CHUNKS))
exemples = CHUNKS[:2] + [c for c in CHUNKS if c["meta"]["type"] == "ligne_tableau"][:2]
for c in exemples:
    print("\n" + "=" * 60); print(c["meta"]["article_titre"], c["meta"]["pages"], c["meta"]["type"]); print(c["text"])

Documents : 6 | Chunks : 653
Counter({'ligne_tableau': 553, 'texte': 69, 'tableau_complet': 31})

Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) [1, 2, 3] texte
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
La base de remboursement est plafonnée au tarif maximal. Données synthétiques.

Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) [1, 2, 3] ligne_tableau
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
CONS-GEN — Libellé : Consultation médecine générale ; Catégorie : CONSULTATION ; Min : 10 000 Ar ; Référence : 20 000 Ar ; Max : 50 000 Ar

Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) [1, 2, 3] ligne_tableau
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
CONS-GEN — Libellé : Consultation médecine générale ; Catégorie : CONSULTATION ; Min : 10 000 Ar ; Référence : 20 000 Ar ; Max : 50 000 Ar

Bare

## 3. Comprendre les expressions hors document : glossaire de synonymes
Si **un** terme d'un groupe apparaît dans un chunk ou dans la question, les autres sont ajoutés au texte **utilisé pour la recherche** (le texte affiché ne change pas).
→ « unité monétaire », « monnaie », « devise » retrouvent « Ariary (MGA) ».
**Complétez `GROUPES`** avec le vocabulaire de vos utilisateurs et celui de vos nouveaux documents.

In [5]:
GROUPES = [
    ["ariary", "mga", "ar", "monnaie", "devise", "unité monétaire", "franc", "argent"],
    ["ticket modérateur", "reste à charge", "part du bénéficiaire", "combien je paie", "ce que je paie", "à ma charge", "participation"],
    ["carence", "délai d'attente", "période d'attente", "temps d'attente", "avant d'être couvert", "avant d'être remboursé"],
    ["plafond", "limite", "maximum", "montant maximal", "montant maximum", "remboursé au maximum"],
    ["entente préalable", "accord préalable", "autorisation préalable", "autorisation avant"],
    ["attestation de prise en charge", "apc", "pec", "accord de prise en charge", "garantie de prise en charge"],
    ["tiers payant", "sans avancer", "dispense d'avance", "carte d'assuré", "ne pas avancer les frais"],
    ["ordonnance", "prescription", "ordonnance périmée", "validité de l'ordonnance"],
    ["bénéficiaire", "assuré", "adhérent", "personne couverte", "ayant droit", "membre de la famille", "qui est couvert"],
    ["conjoint", "époux", "épouse", "mari", "femme", "enfant", "enfants à charge", "famille"],
    ["exclusion", "exclu", "non remboursable", "pas couvert", "non pris en charge", "ce qui n'est pas remboursé"],
    ["fraude", "abus", "falsification", "faux", "tromperie", "contrôle médical"],
    ["réclamation", "contester", "contestation", "recours", "plainte", "litige", "désaccord"],
    ["résiliation", "radiation", "fin du contrat", "arrêt du contrat", "annulation", "quitter"],
    ["prise d'effet", "date de début", "durée du contrat", "tacite reconduction", "renouvellement"],
    ["maternité", "maternite", "accouchement", "césarienne", "grossesse", "naissance", "bébé"],
    ["dentaire", "dent", "dentiste", "prothèse dentaire", "orthodontie", "appareil dentaire"],
    ["optique", "lunettes", "verres", "monture", "vue", "ophtalmologue"],
    ["kinésithérapie", "kinesitherapie", "kiné", "rééducation", "physiothérapie"],
    ["imagerie", "radio", "radiographie", "échographie", "scanner", "irm", "scan"],
    ["laboratoire", "analyses", "analyse de sang", "bilan sanguin", "biologie", "examens biologiques"],
    ["hospitalisation", "hôpital", "clinique", "chirurgie", "opération", "intervention chirurgicale", "séjour"],
    ["pharmacie", "médicament", "médicaments", "pharmacien"],
    ["consultation", "médecin", "docteur", "généraliste", "spécialiste", "rendez-vous"],
    ["données personnelles", "données de santé", "confidentialité", "vie privée", "rgpd", "protection des données"],
    ["contact", "téléphone", "numéro", "email", "e-mail", "joindre", "appeler", "urgence", "urgences"],
    ["statut", "suivi", "dossier", "en cours", "payée", "rejetée", "motif de rejet", "refus", "refusé"],
    ["prestataire conventionné", "établissement conventionné", "réseau de soins", "réseau", "conventionné"],
    ["remboursement hors tiers payant", "j'ai avancé", "avance des frais", "virement", "mobile money"],
    ["pièces justificatives", "documents à fournir", "justificatifs", "papiers", "dossier à fournir"],
    # ➜ ajoutez ici les groupes de synonymes de vos nouveaux documents
]

def norm(s):
    s = unicodedata.normalize("NFD", s.lower().replace("’", "'"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def _motif(t):
    t = norm(t)
    return re.compile(r"(?<!\w)" + re.escape(t) + (r"(?!\w)" if len(t) <= 3 else ""))

GROUPES_N = [[(t, _motif(t)) for t in g] for g in GROUPES]

def expandre(texte):
    n = norm(texte)
    ajouts = []
    for g in GROUPES_N:
        if any(m.search(n) for _, m in g):
            ajouts += [t for t, m in g if not m.search(n)]
    return " ".join(ajouts)

print(expandre("Quelle est l'unité monétaire utilisée ?"))

ariary mga ar monnaie devise franc argent


## 4. Index hybride : embeddings (bge-m3) + BM25

In [6]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import os, pickle, hashlib

EMBED_MODEL = "BAAI/bge-m3"                 # multilingue, très bon en français
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"    # relit les meilleurs candidats

embedder = SentenceTransformer(EMBED_MODEL)
reranker = CrossEncoder(RERANK_MODEL)

STOP = set("le la les un une des du de d l et ou a au aux en dans par pour sur est sont ce ces cette que qui quoi quel quelle quels quelles il elle ils on se sa son ses leur leurs ne pas plus avec sans y".split())

def tokeniser(texte):
    mots = re.findall(r"\w+", norm(texte))
    return [m[:6] for m in mots if m not in STOP and len(m) > 1]   # racinisation légère

CHUNK_VERSION = "v4"   # à changer si vous modifiez la logique de découpage

def construire_index(utiliser_cache=True):
    """À rappeler après tout changement de CHUNKS ou de GROUPES.
    Seuls les documents modifiés (ou le glossaire changé) sont recalculés."""
    global EMB, bm25
    if not CHUNKS:
        raise RuntimeError("CHUNKS est vide : aucun PDF lu. Relancez les sections 1 et 2 après avoir ajouté un PDF.")
    for c in CHUNKS:
        c["search_text"] = c["text"] + "\n" + expandre(c["text"])

    chemin_cache = os.path.join(DOSSIER_PDF, ".index_cache.pkl")
    cache = {}
    if utiliser_cache and os.path.exists(chemin_cache):
        try:
            cache = pickle.load(open(chemin_cache, "rb"))
        except Exception:
            cache = {}
    glos = hashlib.sha1(str(GROUPES).encode()).hexdigest()

    ids_par_doc = defaultdict(list)
    for i, c in enumerate(CHUNKS):
        ids_par_doc[c["meta"]["source"]].append(i)

    EMB = np.zeros((len(CHUNKS), embedder.get_sentence_embedding_dimension()), dtype=np.float32)
    nouveau_cache = {}
    for nom, ids in ids_par_doc.items():
        cle = (DOC_HASH.get(nom), glos, EMBED_MODEL, CHUNK_VERSION, MAX_CHARS, len(ids))
        if nom in cache and cache[nom][0] == cle:
            E, etat = cache[nom][1], "inchangé (cache)"
        else:
            E = embedder.encode([CHUNKS[i]["search_text"] for i in ids], normalize_embeddings=True,
                                show_progress_bar=False, convert_to_numpy=True)
            etat = "recalculé"
        EMB[ids] = E
        nouveau_cache[nom] = (cle, E)
        print(f"  {nom} : {len(ids)} chunks – {etat}")
    try:
        pickle.dump(nouveau_cache, open(chemin_cache, "wb"))
    except Exception as e:
        print("cache non écrit :", e)

    bm25 = BM25Okapi([tokeniser(c["search_text"]) for c in CHUNKS])
    print("Index prêt :", EMB.shape[0], "chunks,", len(DOCUMENTS), "document(s)")

construire_index()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

/tmp/ipykernel_625/3425985724.py:41: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMB = np.zeros((len(CHUNKS), embedder.get_sentence_embedding_dimension()), dtype=np.float32)


  Bareme Reference Actes : 89 chunks – recalculé
  Conditions Generales CONFORT : 36 chunks – recalculé
  Conditions Generales ESSENTIEL : 36 chunks – recalculé
  Conditions Generales PREMIUM : 36 chunks – recalculé
  Guide Reseau Soins Antananarivo : 193 chunks – recalculé
  Guide Reseau Soins National : 263 chunks – recalculé
Index prêt : 653 chunks, 6 document(s)


## 5. Recherche : hybride → fusion RRF → reranking → seuil de confiance

In [7]:
SEUIL_REJET = 0.10   # à calibrer avec les scores de la section 6

# Mot de la question -> mot-clé du nom de fichier. Si la question en cite un seul, la recherche est limitée à ce document.
ALIAS_DOCS = {"essentiel": "ESSENTIEL", "confort": "CONFORT", "premium": "PREMIUM",
              "antananarivo": "Antananarivo", "national": "National", "bareme": "Bareme"}

def detecter_document(question):
    n = norm(question)
    trouves = {v for k, v in ALIAS_DOCS.items() if re.search(r"(?<!\w)" + re.escape(k) + r"(?!\w)", n)}
    return trouves.pop() if len(trouves) == 1 else None

def rechercher(question, top_k=3, n_candidats=20, document=None, verbose=True):
    """document : mot-clé du nom d'un PDF pour limiter la recherche à ce document."""
    if not CHUNKS or "EMB" not in globals() or len(EMB) != len(CHUNKS):
        raise RuntimeError(f"Index absent ou périmé (chunks={len(CHUNKS)}). Relancez les sections 1, 2 puis 4 (construire_index()) dans l'ordre.")
    if document is None:
        document = detecter_document(question)
    q_exp = question + " " + expandre(question)
    autorises = [i for i, c in enumerate(CHUNKS)
                 if document is None or norm(document) in norm(c["meta"]["source"])]
    if not autorises:
        print(f"⚠ Aucun document ne correspond à document={document!r}. Documents : {list(DOCUMENTS)}")
        return [], False
    idx = np.array(autorises)

    d = (embedder.encode([q_exp], normalize_embeddings=True, convert_to_numpy=True) @ EMB[idx].T)[0]
    b = np.array(bm25.get_scores(tokeniser(q_exp)))[idx]

    fusion = defaultdict(float)
    for scores in (d, b):
        for rang, j in enumerate(np.argsort(-scores)[:n_candidats]):
            fusion[int(idx[j])] += 1.0 / (60 + rang + 1)
    candidats = sorted(fusion, key=fusion.get, reverse=True)[:n_candidats]

    bruts = np.array(reranker.predict([(question, CHUNKS[i]["search_text"]) for i in candidats]), dtype=float)
    sc = 1 / (1 + np.exp(-bruts)) if (bruts.min() < 0 or bruts.max() > 1) else bruts

    ordre = np.argsort(-sc)
    resultats = [{"score": float(sc[j]), "chunk": CHUNKS[candidats[j]]} for j in ordre[:top_k]]
    fiable = bool(resultats) and resultats[0]["score"] >= SEUIL_REJET

    if verbose:
        print(f"Question : {question}")
        if not fiable:
            print("⚠ Aucune information fiable trouvée dans les documents.")
        for rang, r in enumerate(resultats, 1):
            m = r["chunk"]["meta"]
            print(f"\n--- Résultat {rang} | score {r['score']:.3f} | {m['article_titre']} (p. {m['pages']}) [{m['type']}]")
            print(r["chunk"]["text"][:700])
    return resultats, fiable

_ = rechercher("Quelle est l'unité monétaire utilisée ?")

Question : Quelle est l'unité monétaire utilisée ?

--- Résultat 1 | score 0.243 | Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) (p. [1, 2, 3]) [ligne_tableau]
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
CONS-ORL — Libellé : Consultation ORL ; Catégorie : CONSULTATION ; Min : 20 000 Ar ; Référence : 30 000 Ar ; Max : 65 000 Ar

--- Résultat 2 | score 0.236 | Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) (p. [1, 2, 3]) [ligne_tableau]
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
IMG-IRM — Libellé : IRM ; Catégorie : IMAGERIE ; Min : 1 000 000 Ar ; Référence : 1 201 500 Ar ; Max : 1 600 000 Ar ; Ordonnance : Oui ; Entente : Oui

--- Résultat 3 | score 0.230 | Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary) (p. [1, 2, 3]) [ligne_tableau]
Bareme Reference Actes – BARÈME DE RÉFÉRENCE DES ACTES ET MÉDICAMENTS (Ariary)
IMG-DOPPLE

In [8]:
_ = rechercher("Combien de temps faut-il attendre avant d'être remboursé pour un accouchement ?", top_k=2)
_ = rechercher("Ma femme et mes enfants sont-ils couverts ?", top_k=2)
_ = rechercher("Quel est le prix du bitcoin ?", top_k=1)

Question : Combien de temps faut-il attendre avant d'être remboursé pour un accouchement ?

--- Résultat 1 | score 0.885 | Conditions Generales ESSENTIEL – Article 5 – Tableau des garanties (p. [1]) [ligne_tableau]
Conditions Generales ESSENTIEL – Article 5 – Tableau des garanties
Maternite — Taux : 90 % ; Plafond annuel : 1 200 000 Ar ; Franchise / facture : Aucune ; Carence : 270 jours

--- Résultat 2 | score 0.876 | Conditions Generales PREMIUM – Article 5 – Tableau des garanties (p. [1]) [ligne_tableau]
Conditions Generales PREMIUM – Article 5 – Tableau des garanties
Maternite — Taux : 100 % ; Plafond annuel : 5 000 000 Ar ; Franchise / facture : Aucune ; Carence : 90 jours
Question : Ma femme et mes enfants sont-ils couverts ?

--- Résultat 1 | score 0.343 | Conditions Generales PREMIUM – Article 3 – Adhésion et bénéficiaires (p. [1]) [texte]
Conditions Generales PREMIUM – Article 3 – Adhésion et bénéficiaires
Peuvent être affiliés : le conjoint et les enfants à charge jusqu'à 21 

## 6. Évaluation (mesurer au lieu de deviner)
`(question, mot-clé du document, [articles attendus])`. Ajoutez vos questions, y compris pour vos nouveaux PDF.

In [ ]:
TESTS = [
    ("Quel est l'objet du contrat ?", "ESSENTIEL", ["1"]),
    ("Quelle est l'unité monétaire utilisée ?", "ESSENTIEL", ["1", "5"]),
    ("Qui sont les bénéficiaires du contrat ?", "ESSENTIEL", ["2", "3"]),
    ("Jusqu'à quel âge mes enfants sont-ils couverts ?", "ESSENTIEL", ["3"]),
    ("Quelle est la durée du contrat ?", "ESSENTIEL", ["4"]),
    ("Quel est le plafond annuel pour l'imagerie ?", "ESSENTIEL", ["5"]),
    ("Quel taux de remboursement pour le dentaire ?", "ESSENTIEL", ["5"]),
    ("Quelle est la franchise en cas d'hospitalisation ?", "ESSENTIEL", ["5"]),
    ("Combien de jours d'attente pour la maternité ?", "ESSENTIEL", ["5", "6"]),
    ("Que se passe-t-il si je me fais soigner pendant la carence ?", "ESSENTIEL", ["6"]),
    ("Quels documents pour une analyse de laboratoire ?", "ESSENTIEL", ["7"]),
    ("Combien de temps une ordonnance est-elle valable ?", "ESSENTIEL", ["7"]),
    ("Que paie le bénéficiaire dans un établissement conventionné ?", "ESSENTIEL", ["8"]),
    ("Sous quel délai le prestataire doit-il envoyer la facture ?", "ESSENTIEL", ["8"]),
    ("Comment être remboursé si j'ai avancé les frais ?", "ESSENTIEL", ["9"]),
    ("Les compléments alimentaires sont-ils remboursés ?", "ESSENTIEL", ["10"]),
    ("L'IRM est-elle prise en charge dans la formule ESSENTIEL ?", "ESSENTIEL", ["10", "7"]),
    ("Comment est détectée la fraude ?", "ESSENTIEL", ["11"]),
    ("Quels sont les motifs de rejet possibles ?", "ESSENTIEL", ["12"]),
    ("Combien de temps pour déposer une réclamation ?", "ESSENTIEL", ["12"]),
    ("Mes données de santé sont-elles protégées ?", "ESSENTIEL", ["13"]),
    ("Quel numéro appeler en urgence ?", "ESSENTIEL", ["14"]),
    ("Qui est l'assureur ?", "ESSENTIEL", ["0"]),
    # --- autres formules
    ("Quel est le plafond dentaire de la formule PREMIUM ?", "PREMIUM", ["5"]),
    ("Quelle est la carence maternité en CONFORT ?", "CONFORT", ["5", "6"]),
    # --- barème
    ("Quel est le tarif de référence d'une consultation de médecine générale ?", "Bareme", ["BARÈME"]),
    ("Quel est le prix de référence du paracétamol ?", "Bareme", ["Médicaments"]),
    # --- guides du réseau
    ("Quelles pharmacies sont ouvertes 24h/24 à Antananarivo ?", "Antananarivo", ["PHARMACIES"]),
    ("Où trouver un dentiste sur rendez-vous à Antananarivo ?", "Antananarivo", ["DENTISTES"]),
    ("Quels laboratoires d'analyses dans le réseau national ?", "National", ["LABORATOIRES"]),
    ("Quelle maternité est ouverte 24h/24 ?", "National", ["MATERNIT"]),
    ("Quel numéro appeler en urgence d'après le guide national ?", "National", ["CONTACTS"]),
    # ➜ ajoutez ici vos propres questions
]

def est_bon(meta, doc, attendus):
    """attendus : numéros d'articles ("5") ou mots du titre de la rubrique ("PHARMACIES")."""
    if norm(doc) not in norm(meta["source"]):
        return False
    for a in attendus:
        if (a.isdigit() and meta["article_numero"] == a) or (not a.isdigit() and norm(a) in norm(meta["article_titre"])):
            return True
    return False

h1 = h3 = 0
for q, doc, attendus in TESTS:
    res, ok = rechercher(q, top_k=3, document=doc, verbose=False)
    if not res:
        print("❌ aucun résultat", q); continue
    bons = [est_bon(r["chunk"]["meta"], doc, attendus) for r in res]
    h1 += bons[0]; h3 += any(bons)
    print("✅" if bons[0] else "🟡" if any(bons) else "❌", f"{res[0]['score']:.2f}", q,
          "→", [f"{r['chunk']['meta']['source'][:12]}:{r['chunk']['meta']['article_numero']}" for r in res])

print(f"\nhit@1 = {h1}/{len(TESTS)}   hit@3 = {h3}/{len(TESTS)}")

✅ 0.97 Quel est l'objet du contrat ? → ['Conditions G:1', 'Conditions G:2', 'Conditions G:13']
✅ 0.22 Quelle est l'unité monétaire utilisée ? → ['Conditions G:1', 'Conditions G:5', 'Conditions G:5']
✅ 0.80 Qui sont les bénéficiaires du contrat ? → ['Conditions G:2', 'Conditions G:3', 'Conditions G:1']
✅ 0.43 Jusqu'à quel âge mes enfants sont-ils couverts ? → ['Conditions G:3', 'Conditions G:2', 'Conditions G:6']
✅ 0.64 Quelle est la durée du contrat ? → ['Conditions G:4', 'Conditions G:3', 'Conditions G:1']
✅ 0.94 Quel est le plafond annuel pour l'imagerie ? → ['Conditions G:5', 'Conditions G:5', 'Conditions G:2']
✅ 0.91 Quel taux de remboursement pour le dentaire ? → ['Conditions G:5', 'Conditions G:5', 'Conditions G:2']
✅ 0.59 Quelle est la franchise en cas d'hospitalisation ? → ['Conditions G:5', 'Conditions G:5', 'Conditions G:5']
✅ 0.82 Combien de jours d'attente pour la maternité ? → ['Conditions G:5', 'Conditions G:5', 'Conditions G:7']
✅ 0.31 Que se passe-t-il si je me fais soi

## 7. Générer la réponse finale (le RAG complet)
La recherche retrouve les passages ; **un LLM** rédige ensuite la réponse en s'appuyant uniquement dessus.
- **A. Modèle local gratuit** (Qwen2.5 sur le GPU Colab) : aucune clé nécessaire.
- **B. API Claude** : meilleure qualité, nécessite une clé.

In [1]:
def construire_messages(question, top_k=5, document=None):
    resultats, fiable = rechercher(question, top_k=top_k, document=document, verbose=False)
    if not fiable:
        return None, None
    groupes = {}                      # une entrée par (document, rubrique)
    for r in resultats:
        c = r["chunk"]; m = c["meta"]
        g = groupes.setdefault((m["source"], m["article_titre"]), {"meta": m, "parent": c["parent"], "textes": []})
        corps = c["text"].split("\n", 1)[1] if "\n" in c["text"] else c["text"]
        if corps not in g["textes"]:
            g["textes"].append(corps)
    contextes, sources = [], []
    for g in groupes.values():
        m = g["meta"]
        pages = ", ".join(str(p) for p in m["pages"])
        # article court -> en entier ; longue liste (guide, barème) -> seulement les lignes trouvées
        corps = g["parent"] if len(g["parent"]) <= 3000 else m["article_titre"] + "\n" + "\n".join(g["textes"])
        contextes.append(f"[{m['article_titre']} – p. {pages}]\n{corps}")
        sources.append(f"{m['article_titre']} (p. {pages})")
    systeme = (
        "Tu es l'assistant de PrediCare TPA. Réponds en français, de façon claire et concise, "
        "UNIQUEMENT à partir des extraits fournis. N'invente rien. "
        "Les extraits peuvent venir de plusieurs documents (formules ESSENTIEL, CONFORT, PREMIUM, barème, guides du réseau) : "
        "indique toujours de quel document vient l'information. Si la question ne précise pas la formule "
        "et que les valeurs diffèrent, donne la valeur de chaque formule. "
        "Si l'information ne figure pas dans les extraits, dis-le clairement. "
        "Termine par la source, par exemple : (Source : Conditions Generales ESSENTIEL, Article 5)."
    )
    utilisateur = "EXTRAITS :\n\n" + "\n\n".join(contextes) + f"\n\nQUESTION : {question}"
    return (systeme, utilisateur), sources

REPONSE_ABSENTE = "Je n'ai pas trouvé cette information dans les documents disponibles."

In [ ]:
# --- Option A : modèle local gratuit (Qwen2.5 sur GPU Colab) ---
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "Qwen/Qwen2.5-3B-Instruct"      # plus léger : "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16, device_map="auto")

def generer_local(systeme, utilisateur, max_tokens=300):
    msgs = [{"role": "system", "content": systeme}, {"role": "user", "content": utilisateur}]
    # On génère le dictionnaire d'encodage avec les tenseurs PyTorch
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    # On extrait spécifiquement le tenseur d'identifiants de tokens (input_ids)
    ids = inputs["input_ids"].to(llm.device)
    with torch.no_grad():
        sortie = llm.generate(ids, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(sortie[0][ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# --- Option B : API Claude (décommentez et renseignez la clé) ---
# !pip install -q anthropic
# import os, anthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# client = anthropic.Anthropic()
# def generer_claude(systeme, utilisateur, max_tokens=400):
#     r = client.messages.create(model="claude-sonnet-5-5", max_tokens=max_tokens,
#                                system=systeme, messages=[{"role": "user", "content": utilisateur}])
#     return r.content[0].text

In [ ]:
GENERATEUR = generer_local      # remplacez par generer_claude si vous utilisez l'option B

def repondre(question, top_k=5, document=None, afficher_sources=True):
    messages, sources = construire_messages(question, top_k, document)
    if messages is None:
        reponse, sources = REPONSE_ABSENTE, []
    else:
        reponse = GENERATEUR(*messages)
    print(f"❓ {question}\n\n💬 {reponse}")
    if afficher_sources and sources:
        print("\n📚 Passages consultés :", " | ".join(sources))
    return reponse

# Lancement du test
repondre("Combien de temps faut-il attendre pour la maternité ?")
print("\n" + "-" * 70 + "\n")
repondre("Quelle est l'unité monétaire utilisée ?")
print("\n" + "-" * 70 + "\n")
repondre("Quel est le prix du bitcoin ?")

## 8. Ajouter un nouveau document
1. Déposez le PDF dans `DOSSIER_PDF`.
2. Relancez les sections **1 → 4** (lecture, chunking, glossaire, index). Les modèles déjà chargés ne sont pas rechargés si vous relancez uniquement les cellules de lecture, de chunking et `construire_index()`.
3. Ajoutez au glossaire les termes du nouveau document, et quelques questions dans `TESTS`.
4. Pour interroger un seul document : `repondre("ma question", document="mot-clé du nom du PDF")`.

*Limites : un PDF scanné (image) demande un OCR préalable ; un document sans titres « Article N – … » est découpé par paragraphes (adapter `RE_TITRE` si sa numérotation est différente).*